# Artistic Telephone: instructor reveal

Collect PNGs from every round in group folders, such as `class_handoffs/Group_A`, `class_handoffs/Group_B`, etc. You can include both original Telephone handoffs **with** embedded prompts and ordinary PNG files **without** them.

The gallery uses valid chain filenames (for example `01.png`, `05_01.png`) to show the image progression. Only Telephone-exported PNGs contain automatically recoverable prompt records. Ordinary PNGs remain visible with **Prompt unavailable**; the notebook cannot reconstruct an unsaved prompt from the image alone.

If you have the original prompt for an ordinary PNG, you can enter it separately in the `PROMPTS` dictionary below. This is clearly labelled as a manually supplied prompt, not a decoded record.

No API account or generation is needed. Keep `telephone_*.py` helpers beside this notebook. Use the shared requirements. Works online with screen sharing or in person with projection. Keep practice and live images in separate folders.


In [ ]:
from pathlib import Path
from IPython.display import HTML, FileLink, display
from telephone_gallery import scan, render

## 1. Choose the downloaded folder

Set `FOLDER` to a directory containing `Group_A`, `Group_B`, etc. For ordinary PNGs, the folder and filename establish their position. Invalid names and duplicate positions are reported.

Leave `FALLBACK_SESSION` and `FALLBACK_MODE` blank to match a uniquely identified embedded session/mode automatically. If all images are ordinary PNGs, you can enter the class session name and `live` or `practice` yourself; otherwise they will appear under `unrecorded`.

If you know a missing prompt, enter it in `PROMPTS`, keyed by the PNG's relative path using forward slashes, e.g. `"Group_A/01.png"`. No prompts are invented.


In [ ]:
FOLDER = ""
FALLBACK_SESSION = ""   # Optional: e.g. "h26_w5" for ordinary PNGs
FALLBACK_MODE = ""      # Optional: "live" or "practice" for ordinary PNGs
PROMPTS = {
    # "Group_A/01.png": "Prompt supplied separately by the student",
    # "Group_A/05_01.png": "Another known prompt",
}
records, warnings = {}, []
if FOLDER:
    records, warnings = scan(
        FOLDER, fallback_session=FALLBACK_SESSION,
        fallback_mode=FALLBACK_MODE, prompt_overrides=PROMPTS,
    )
    entries = [entry for entry in records.values() if entry is not None]
    embedded = sum(e["record"]["record_source"] == "embedded" for e in entries)
    supplied = sum(e["record"]["record_source"] == "manual" for e in entries)
    unavailable = sum(e["record"]["record_source"] == "missing" for e in entries)
    print("Groups found:", sorted({key[2] for key in records}))
    print(f"Images: {len(entries)} | embedded prompts: {embedded} | "
          f"manually supplied prompts: {supplied} | prompts unavailable: {unavailable}")
    for warning in warnings:
        print("Check:", warning)
else:
    print("Set FOLDER to the downloaded class folder, then rerun.")


## 2. Show the progression

Each chain follows the filenames `01.png`, `01_02.png`, `01_02_03.png`, etc. All valid PNGs remain visible in their positions even if they have no hidden record.

Use **Reveal prompt and settings** for embedded records, or **Reveal separately supplied prompt** when you have manually added one. Plain PNGs without a known prompt display **Prompt unavailable**. A missing prompt is not an image error, and is not evidence of a failed generation.

Use the progression to discuss interpretation, changes and visual drift; it is not an evaluation score.


In [ ]:
gallery_html = render(records, warnings)
display(HTML(gallery_html))

## 3. Save the gallery
Set SAVE_GALLERY to True to export one HTML file containing images and prompts. Open it in a browser without network access. A full-class gallery may be large. Share only after the game: the prompt reveal controls are not access protection.

In [ ]:
SAVE_GALLERY = False
if SAVE_GALLERY and records:
    target = Path("outputs") / "telephone_reveal.html"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(gallery_html, encoding="utf-8")
    display(FileLink(str(target)))
else:
    print("Load images and set SAVE_GALLERY to True when ready.")

## If a prompt cannot be revealed

Check whether you received the **selected handoff PNG** exported from notebook 02, rather than the raw image saved by Replicate, a screenshot, or a file that was resized or converted. Those files can look identical but will not include the Telephone prompt record. For future games, use **Save selected image for passing** and pass the original PNG unchanged.

If the record is missing, the gallery still displays the image under its valid filename. If you have a reliable copy of the student's original prompt, add it to `PROMPTS` in section 1 and rescan. If you do not, leave it unavailable; do not infer it from the image.

The pixel encoding discourages casual prompt inspection but is not encryption, authentication, or tamper-proof storage.
